### Linki między filmami, reżyserami i recenzjami

Na koniec dodamy linki umożliwiające łatwą nawigację między widokami – np. klikając nazwę filmu w recenzji, przejdziemy do jego widoku; klikając reżysera – do jego profilu itd.

#### Plik: `views.py`

In [ ]:
class MovieListViewWithLinks(ListView):
    model = Movie
    template_name = "movie_list_with_links.html"
class ReviewListViewWithLinks(ListView):
    model = Review
    template_name = "review_list_with_links.html"

#### Plik: `urls.py`

In [ ]:
from movies.views import MovieListViewWithLinks, ReviewListViewWithLinks

urlpatterns += [
    path("movielinks/", MovieListViewWithLinks.as_view()),
    path("reviewlinks/", ReviewListViewWithLinks.as_view()),
]

#### Szablon: `templates/movie_list_with_links.html`

In [ ]:
<h1>Filmy</h1>
<ul>
{% for movie in object_list %}
    <li>
        <strong>{{ movie.title }}</strong> — reżyser:
        <a href="{% url 'director-detail' movie.director.id %}">{{ movie.director.last_name }}</a>
        <br>
        <a href="{% url 'movie-detail' movie.id %}">Zobacz szczegóły filmu</a>
    </li>
{% endfor %}
</ul>

#### Szablon: `templates/review_list_with_links.html`

In [ ]:
<h1>Recenzje</h1>
<ul>
{% for review in object_list %}
    <li>
        <strong>{{ review.author }}</strong> o
        <a href="{% url 'movie-detail' review.movie.id %}">{{ review.movie.title }}</a>:<br>
        {{ review.text }}
    </li>
{% endfor %}
</ul>

### Uwaga: Błąd `NoReverseMatch` i puste pola `ForeignKey`

Jeśli podczas wyświetlania listy filmów pojawi się błąd:

`NoReverseMatch: Reverse for ‘director-detail’ with arguments ‘(’’,)’ not found.`

oznacza to, że Django próbowało wygenerować adres URL do szczegółowego widoku reżysera (`director-detail`), ale zabrakło wartości `id`. Dzieje się tak, gdy któryś z filmów **nie ma przypisanego reżysera** (pole `director` jest puste, czyli `None`).

Aby uniknąć tego błędu, w szablonie należy sprawdzić, czy dany obiekt istnieje:

```django
{% if movie.director %}
    <a href="{% url 'director-detail' movie.director.id %}">{{ movie.director.name }}</a>
{% else %}
    brak danych
{% endif %}
```

To zabezpieczenie powinno być stosowane zawsze, gdy pole `ForeignKey` może być puste.

# Django - Użytkownicy aplikacji 1
*[Mikołaj Leszczuk](mailto:mikolaj.leszczuk@agh.edu.pl), [Agnieszka Rudnicka](mailto:rudnicka@agh.edu.pl)*

* Django - wbudowane mechanizmy uwierzytelnienia
* Sprawdzenie konfiguracji projektu
* Podstawowe widoki użytkowników
  * Otwórzmy przeglądarkę
  * Szablon strony logowania
  * Logujemy się
  * Ćwiczenie - strona profilu zalogowanego użytkownika
  * Rozwiązanie

Nasza dotychczasowa wersja strony posiada prosty model filmów. Niestety, żeby dodać nową pozycję lub edytować
istniejącą trzeba się zalogować do panelu administracyjnego. Gdybyśmy chcieli dodać możliwość oceniania filmów i pisania recenzji, panel administracyjny to za mało.

Dodajmy więc możliwość rejestracji i logowania się użytkowników, którzy nie są administratorami.

## Django - wbudowane mechanizmy uwierzytelnienia

Zgodnie z dokumentacją Django: https://docs.djangoproject.com/en/5.2/topics/auth/ framework dostarcza nam potrzebny
szkielet kodu na start.

## Sprawdzenie konfiguracji projektu

Na początek sprawdźmy, czy mamy dodaną aplikację `django.contrib.auth` oraz `django.contrib.contenttypes` do
`INSTALLED_APPS` w ustawieniach naszego projektu ([`goodmovies/settings.py`](http://localhost:8888/edit/goodmovies/settings.py)).

```python
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',  # <-- tu jest
    'django.contrib.contenttypes',  # <-- i tu
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    'movies',
]
```

Następnie sprawdźmy czy mamy `SessionMiddleware` oraz `AuthenticationMiddleware` w naszych ustawieniach
`MIDDLEWARE`. Znajdziemy je również w ustawieniach naszego projektu ([`goodmovies/settings.py`](http://localhost:8888/edit/goodmovies/settings.py)).

```python
MIDDLEWARE = [
    'django.middleware.security.SecurityMiddleware',
    'django.contrib.sessions.middleware.SessionMiddleware',  # <-- tu
    'django.middleware.common.CommonMiddleware',
    'django.middleware.csrf.CsrfViewMiddleware',
    'django.contrib.auth.middleware.AuthenticationMiddleware',  # <-- i tu
    'django.contrib.messages.middleware.MessageMiddleware',
    'django.middleware.clickjacking.XFrameOptionsMiddleware',
]
```

Następnie zalecane jest zaaplikowanie migracji, które stworzą w bazie danych tabele na użytkowników:

In [3]:
!python3 manage.py makemigrations

No changes detected


In [4]:
!python3 manage.py migrate

Operations to perform:
  Apply all migrations: admin, auth, contenttypes, movies, sessions
Running migrations:
  No migrations to apply.


Dla nas jednak nic nowego się nie powinno wykonać, bo mieliśmy już zainstalowaną aplikację użytkowników od początku - i tabele użytkowników zostały stworzone razem z tabelami filmów i innymi.

Gdybyśmy mieli źle skonfigurowaną aplikację użytkowników nie moglibyśmy się zalogować do panelu administracyjnego.

## Podstawowe widoki użytkowników

Teraz kiedy upewniliśmy się, że mamy poprawnie skonfigurowaną aplikację do obsługi użytkowników pora zobaczyć co
potrafi! Będziemy się opierać głównie na przykładach z [oficjalnej dokumentacji](https://docs.djangoproject.com/en/5.2/topics/auth/default/), nie ma tu żadnej "magii" :)

Otwórzmy nasz plik [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py), gdzie znajdują się wszystkie obsługiwane ścieżki.

Django dostarcza kilka widoków, które zajmują się podstawowym zarządzaniem sesją użytkownika. Mamy tutaj:

* logowanie

* wylogowanie

* zmianę hasła

* reset hasła (wymaga dodatkowej konfiguracji m.in. wysyłki email)

Wszystkie te widoki znajdują się w paczce `django.contrib.auth.urls`. Dołączmy je do naszej aplikacji dodając
następujący kod do pliku [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py) do listy `urlpatterns`:

```python
urlpatterns += [
    path('accounts/', include('django.contrib.auth.urls')),
]
```

Musimy jeszcze zamieścić import funkcji `include`. Dodajmy więc w pierwszych linijkach pliku:

```python
from django.urls import include
```

Funkcja `include` pozwala nam zaimportować i dołączyć do routingu naszej aplikacji wszystkie widoki zdefiniowane w
`django.contrib.auth.urls`. Dzięki temu nasz projekt użyje już raz zaprogramowanych przez twórców Django widoków.

W podobny sposób możemy dołączać widoki innych aplikacji/bibliotek dodanych do projektu lub innych naszych aplikacji. Czasem bowiem plik [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py) dzieli się na kilka mniejszych, żeby nie były zbyt długie/przytłaczające :)

### Otwórzmy przeglądarkę

Aktywujmy nasze środowisko wirtualne, jeśli jeszcze tego nie zrobiliśmy lub nasze IDE nas nie wyręczyło. Uruchommy
naszą aplikację poleceniem:
```zsh
python3 manage.py runserver
```
Następnie otwórzmy przeglądarkę na standardowym adresie http://127.0.0.1:8000/.

Jeśli działa, to sprawdzamy dalej. Dodaliśmy widoki logowania pod adresem `/accounts/`, więc przejdźmy w przeglądarce pod http://127.0.0.1:8000/accounts/.

Blisko, pod tym adresem nic nie ma, ale widzimy, że na liście "dostępnych" URLi znajdują się nowe widoki, w tym
`accounts/login/`.

Przejdźmy pod http://127.0.0.1:8000/accounts/login/

To już znamy :) O ile wbudowana appka Django dostarcza nam widoków/logiki, to nie narzuca nam szablonów HTML.
Jednak w [dokumentacji można znaleźć startowy kawałek kodu](https://docs.djangoproject.com/en/5.2/topics/auth/default/#all-authentication-views).

### Szablon strony logowania

Utwórzmy zatem plik [`movies/templates/registration/login.html`](http://localhost:8888/edit/movies/templates/registration/login.html):

In [5]:
!mkdir movies/templates/registration

In [6]:
!touch movies/templates/registration/login.html

Ponieważ to szablon HTML - musi się on znajdować w katalogu `templates` naszej aplikacji. Django zakłada, że HTML będzie w podkatalogu `registration`.

Oto snippet kodu, którego użyjemy:

```django
{% extends "base.html" %}

{% block content %}

    {% if form.errors %}
        <p>Your username and password didn't match. Please try again.</p>
    {% endif %}

    {% if next %}
        {% if user.is_authenticated %}
            <p>Your account doesn't have access to this page. To proceed,
            please login with an account that has access.</p>
        {% else %}
            <p>Please login to see this page.</p>
        {% endif %}
    {% endif %}

    <form method="post" action="{% url 'login' %}">
        {% csrf_token %}
        <table>
            <tr>
                <td>{{ form.username.label_tag }}</td>
                <td>{{ form.username }}</td>
            </tr>
            <tr>
                <td>{{ form.password.label_tag }}</td>
                <td>{{ form.password }}</td>
            </tr>
        </table>

        <input type="submit" value="login">
        <input type="hidden" name="next" value="{{ next }}">
    </form>

{% endblock %}
```

Jeśli plik został dobrze nazwany, to po odświeżeniu strony w przeglądarce powinniśmy ujrzeć formularz logowania.

Notka: Warto wcześniej wejść do panelu administracyjnego (http://127.0.0.1:8000/admin/) i się z niego wylogować, bo Django może pamiętać sesję z
poprzednich zajęć i nie ujrzymy formularza logowania.

### Logujemy się

Możemy oczywiście skorzystać z naszych danych logowania do panelu administracyjnego. Jeśli ktoś nie ma konta, to: 
```zsh
python3 manage.py createsuperuser
```

### Ćwiczenie - strona profilu zalogowanego użytkownika

Jednym z rozwiązań jest dodanie widoku profilu aktualnie zalogowanego użytkownika.

Inne to zmiana adresu przekierowania https://docs.djangoproject.com/en/5.2/ref/settings/#login-redirect-url. Czyli np ustawienie `LOGIN_REDIRECT_URL="/"` w naszym pliku [`goodmovies/settings.py`](http://localhost:8888/edit/goodmovies/settings.py).

Skupmy się jednak na tym pierwszym, w naszym pliku [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py) potrzebujemy dodać nowy widok, który obsłuży zapytania
kierowane na adres `accounts/profile/` naszej aplikacji. Dodajmy więc odpowiednią ścieżkę:

```python
urlpatterns += [
    path(
        'accounts/profile/',
        views.profile_view, 
        name='user_profile'
    ),
]
```

do istniejących już w liście `urlpatterns`.

Taka funkcja `profile_view` oczywiście jeszcze nie istnieje, musimy ją napisać :)

Spróbujcie napisać widok i podpiąć do niego szablon, który wyświetla nazwę zalogowanego użytkownika oraz datę
ostatniego logowania. Podpowiedź:

Datę ostatniego logowania wyświetlimy w HTML:

```django
<p>Ostatnie logowanie {{ request.user.last_login }}</p>
```

Spis wszystkich pól modelu użytkownika znajdziecie w dokumentacji: https://docs.djangoproject.com/en/5.2/ref/contrib/auth/#django.contrib.auth.models.User

Przykładowy wygląd strony profilowej:

<h1>Witaj miklesz!</h1>
Ostatnie logowanie May 15, 2077, 9:41 a.m.

### Rozwiązanie

#### Plik: [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py)
Dodajemy ścieżkę do widoku profilu użytkownika.

```python
# W liście urlpatterns:
urlpatterns += [
    path(
        'accounts/profile/',
        views.profile_view, 
        name='user_profile'
    ),
]
```

#### Plik: [`movies/views.py`](http://localhost:8888/edit/movies/views.py)
Tworzymy widok profilu zalogowanego użytkownika.

```python
def profile_view(request):
    return render(request, "profile.html")
```

#### Plik: [`movies/templates/profile.html`](http://localhost:8888/edit/movies/templates/profile.html)
Szablon strony profilowej użytkownika.

In [7]:
!touch movies/templates/profile.html

```django
<h1>Witaj {{ request.user.username }}!</h1>
<p>Ostatnie logowanie {{ request.user.last_login }}</p>
```

#### Uwaga
Jeśli po zalogowaniu Django przekierowuje gdzieś indziej, dodaj do `settings.py` wpis:
```python
LOGIN_REDIRECT_URL = '/accounts/profile/'
```
Można też ustawić np. `/` albo dowolną inną stronę jako domyślną po zalogowaniu.

# Django - Użytkownicy aplikacji 2
*[Mikołaj Leszczuk](mailto:mikolaj.leszczuk@agh.edu.pl), [Agnieszka Rudnicka](mailto:rudnicka@agh.edu.pl)*

* Rejestracja użytkowników
  * URL
  * Widok (logika)
    * Dlaczego sprawdzamy `request.method == "POST"`?
    * Szablony HTML
    * Zapisywanie danych z formularza
    * Finalny widok rejestracji
* Dalsze kroki
  * Ćwiczenie
  * Rozwiązanie
* Rozszerzamy interfejs użytkownika
  * Ćwiczenie
  * Rozwiązanie
* Źródła i materiały pomocnicze

## Rejestracja użytkowników

Skoro możemy się już zalogować pora na dodanie rejestracji. Potrzebujemy kilku rzeczy:

1. szablonu HTML z formularzem rejestracji

2. widoku, który przetworzy dane nowego użytkownika i stworzy mu konto

3. szablonu HTML potwierdzającego poprawne założenie konta

Oczywiście metod na zaimplementowanie przebiegu rejestracji jest mnóstwo, to tylko jedna z nich :)

### URL

Standardowo potrzebujemy dodać nową ścieżkę do [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py), tym razem niech to będzie:

```python
urlpatterns += [
    path(
        'accounts/signup/',
        views.user_signup,
        name="user_signup"
    ),
]
```